## Random Forest Implementation

In [1]:
# importing libraries and data
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
def calc_entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    return -np.sum(proportions * np.log2(proportions))

def information_gain(X_column, y, threshold):

    parent_entropy = calc_entropy(y)

    left_mask = X_column <= threshold
    right_mask = X_column > threshold
    y_left, y_right = y[left_mask], y[right_mask]

    if len(y_left) == 0 or len(y_right) == 0:
        return 0

    n = len(y)
    weighted_entropy = (len(y_left)/n) * calc_entropy(y_left) + \
                       (len(y_right)/n) * calc_entropy(y_right)

    return parent_entropy - weighted_entropy

def best_split(X, y, n_features=None):
    best_gain = 0
    best_feature = None
    best_threshold = None

    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            gain = information_gain(X[:, feature], y, threshold)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold

class Node:
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction

def build_tree(X, y, max_depth=10, depth=0, n_features=None):

    if len(np.unique(y)) == 1:
        return Node(prediction=y[0])
    if depth >= max_depth:
        return Node(prediction=np.bincount(y).argmax())
    if len(y) < 2:
        return Node(prediction=np.bincount(y).argmax())

    feature, threshold = best_split(X, y)

    if feature is None:
        return Node(prediction=np.bincount(y).argmax())

    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    left = build_tree(X[left_mask], y[left_mask], max_depth, depth+1)
    right = build_tree(X[right_mask], y[right_mask], max_depth, depth+1)

    return Node(feature=feature, threshold=threshold, left=left, right=right)

def predict_one(node, x):
    if node.prediction is not None:
        return node.prediction
    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(node, X):
    return np.array([predict_one(node, x) for x in X])

In [3]:
# random forest
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from scipy import stats

class RandomForestFromScratch:
    def __init__(self, n_trees=10, max_depth=5):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):

            indices = np.random.choice(len(X), size=len(X), replace=True)
            X_sample, y_sample = X[indices], y[indices]

            tree = build_tree(X_sample, y_sample,
                            max_depth=self.max_depth,
                            n_features=int(np.sqrt(X.shape[1])))
            self.trees.append(tree)

    def predict(self, X):
        all_predictions = np.array([predict(tree, X) for tree in self.trees])
        return stats.mode(all_predictions, axis=0)[0].flatten()

# training
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

single_tree = build_tree(X_train, y_train, max_depth=5)
single_tree_acc = np.mean(predict(single_tree, X_test) == y_test)
print(f"Single tree accuracy:  {single_tree_acc * 100:.2f}%")

rf = RandomForestFromScratch(n_trees=100, max_depth=5)
rf.fit(X_train, y_train)
rf_acc = np.mean(rf.predict(X_test) == y_test)
print(f"Random forest accuracy: {rf_acc * 100:.2f}%")

print("\nAccuracy vs number of trees:")
for n in [1, 5, 10, 25, 50, 100]:
    rf = RandomForestFromScratch(n_trees=n, max_depth=5)
    rf.fit(X_train, y_train)
    acc = np.mean(rf.predict(X_test) == y_test)
    print(f"n_trees={n:3d}  →  {acc * 100:.2f}%")

Single tree accuracy:  100.00%
Random forest accuracy: 100.00%

Accuracy vs number of trees:
n_trees=  1  →  100.00%
n_trees=  5  →  100.00%
n_trees= 10  →  100.00%
n_trees= 25  →  100.00%
n_trees= 50  →  100.00%
n_trees=100  →  100.00%


In [ ]:
# comparison with scikit-learn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_hard, y_hard = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=42
)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hard, y_hard, test_size=0.2, random_state=42)

single_tree = build_tree(X_train_h, y_train_h, max_depth=5)
single_acc = np.mean(predict(single_tree, X_test_h) == y_test_h)
print(f"Single tree accuracy: {single_acc * 100:.2f}%")

print("\nAccuracy vs number of trees:")
for n in [1, 5, 10, 25, 50, 100]:
    rf = RandomForestFromScratch(n_trees=n, max_depth=5)
    rf.fit(X_train_h, y_train_h)
    acc = np.mean(rf.predict(X_test_h) == y_test_h)
    print(f"n_trees={n:3d}  →  {acc * 100:.2f}%")

Single tree accuracy: 82.00%

Accuracy vs number of trees:
n_trees=  1  →  80.50%
n_trees=  5  →  82.00%
n_trees= 10  →  84.50%
n_trees= 25  →  90.50%
n_trees= 50  →  89.00%
n_trees=100  →  88.00%
